In [1]:
from pymongo import MongoClient

# 1. Conectar al contenedor de Mongo
client = MongoClient('mongodb://mongodb:27017/')

# 2. Crear (o usar) la base de datos y la colección
db = client['PruebaYapo'] 
coleccion = db['G2_PruebaRealState'] # Ideal el nombre del grupo ejem: 'G_Ecommerce_scraping'

print("Conexión exitosa a MongoDB")

Conexión exitosa a MongoDB


In [2]:
import time
import csv
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

print("🚀 Iniciando extracción de propiedades en Yapo.cl (Coquimbo - La Serena)...")

# 1. Configuración Headless y Anti-Detección
options = Options()
options.add_argument("--headless=new") # Ejecuta en segundo plano
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920,1080")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

driver = webdriver.Chrome(options=options)

# Lista para guardar los datos antes de pasarlos al CSV
datos_extraidos = []

try:
    # 2. Navegación
    url = "https://www.yapo.cl/bienes-raices-alquiler-apartamentos/chile-es-coquimbo?_gl=1*xrdyl5*_gcl_au*MjQxMDY4NDMuMTc3NjU1NTI1MQ.."
    print(f"🌐 Navegando a: {url}")
    driver.get(url)

    # 3. Esperar carga inicial
    print("⏳ Esperando carga de los anuncios...")
    SELECTOR_CONTENEDOR = ".d3-ad-tile__content"
    
    WebDriverWait(driver, 15).until(
        EC.presence_of_all_elements_located((By.CSS_SELECTOR, SELECTOR_CONTENEDOR))
    )
    
    # --- SCROLL PARA CARGAR IMÁGENES Y DATOS REZAGADOS ---
    print("📜 Deslizando la página para renderizar todo el contenido...")
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(5) 

    # 4. Capturar bloques principales
    bloques = driver.find_elements(By.CSS_SELECTOR, SELECTOR_CONTENEDOR)
    print(f"✅ Se detectaron {len(bloques)} bloques en total en esta página.\n")
    print("-" * 50)

    # 5. Iteración y Extracción
    propiedades_validas = 0
    
    # Quitamos el límite de [:10] para que extraiga todos los de la página
    for bloque in bloques: 
        try:
            # Extracción de Textos Directos
            titulo = bloque.find_element(By.CSS_SELECTOR, ".d3-ad-tile__title").text.strip()
            precio_texto = bloque.find_element(By.CSS_SELECTOR, ".d3-ad-tile__price").text.strip()
            ubicacion = bloque.find_element(By.CSS_SELECTOR, ".d3-ad-tile__location").text.strip()
            
            # El vendedor a veces puede no estar, usamos un try-except interno rápido
            try:
                vendedor = bloque.find_element(By.CSS_SELECTOR, ".d3-ad-tile__seller span").text.strip()
            except:
                vendedor = "Particular / No especificado"

            try:
                descripcion = bloque.find_element(By.CSS_SELECTOR, ".d3-ad-tile__short-description").text.strip()
            except:
                descripcion = "Sin descripción"
            
            # Extracción de URL 
            url_anuncio = bloque.find_element(By.CSS_SELECTOR, "a.d3-ad-tile__description").get_attribute("href")

            # Extracción de Detalles (Dormitorios, Estacionamientos, Baños)
            detalles = bloque.find_elements(By.CSS_SELECTOR, ".d3-ad-tile__details-item")
            dormitorios = detalles[0].text.strip() if len(detalles) > 0 else "0"
            estacionamientos = detalles[1].text.strip() if len(detalles) > 1 else "0"
            banos = detalles[2].text.strip() if len(detalles) > 2 else "0"
            
            # Filtro antiblancos (evita banners de publicidad)
            if not titulo or not precio_texto:
                continue
            
            # Limpieza del precio para la Base de Datos
            precio_limpio = precio_texto.replace("$", "").replace(".", "").strip()
            precio_numerico = float(precio_limpio) if precio_limpio.isdigit() else 0.0

            propiedades_validas += 1
            print(f"🏠 PROPIEDAD #{propiedades_validas}: {titulo} | Precio BD: {precio_numerico}")
            
            # Guardamos el registro en nuestro diccionario
            datos_extraidos.append({
                "Titulo": titulo,
                "Precio_Original": precio_texto,
                "Precio_Numerico": precio_numerico,
                "Ubicacion": ubicacion,
                "Vendedor": vendedor,
                "Dormitorios": dormitorios,
                "Banos": banos,
                "Estacionamientos": estacionamientos,
                "Descripcion": descripcion,
                "Enlace": url_anuncio
            })
            
        except Exception as e:
            # Captura errores silenciosos si cambia la estructura de un anuncio específico
            pass

    print("-" * 50)
    print(f"📊 Total de propiedades válidas extraídas: {propiedades_validas}")

    # 6. Exportación a CSV
    if datos_extraidos:
        archivo_csv = 'inmuebles_yapo_coquimbo.csv'
        print(f"💾 Guardando los datos en {archivo_csv}...")
        
        # Obtenemos los nombres de las columnas de las llaves del primer diccionario
        columnas = datos_extraidos[0].keys()
        
        with open(archivo_csv, 'w', newline='', encoding='utf-8-sig') as archivo:
            writer = csv.DictWriter(archivo, fieldnames=columnas)
            writer.writeheader()
            writer.writerows(datos_extraidos)
            
        print("✅ ¡Archivo CSV generado con éxito!")

except Exception as e:
    print(f"🚨 Error crítico durante la ejecución: {e}")

finally:
    # 7. Cierre seguro del WebDriver
    print("🧹 Cerrando el navegador...")
    driver.quit()
    print("✨ Proceso completado.")

🚀 Iniciando extracción de propiedades en Yapo.cl y carga a MongoDB...
✅ Conexión exitosa a MongoDB
🌐 Navegando a: https://www.yapo.cl/bienes-raices-alquiler-apartamentos/chile-es-coquimbo?_gl=1*xrdyl5*_gcl_au*MjQxMDY4NDMuMTc3NjU1NTI1MQ..
⏳ Esperando carga de los anuncios...
📜 Deslizando la página para renderizar todo el contenido...
✅ Se detectaron 30 bloques en total en esta página.

--------------------------------------------------
🏠 PROPIEDAD #1: Arriendo año corrido, amoblado, 2D 2B “Cumbres de Peñuelas” Coquimbo | Precio: 600000.0
🏠 PROPIEDAD #2: DEPARTAMENTO Avenida Las Higueras con Anibal Pinto | Precio: 0.0
🏠 PROPIEDAD #3: Arriendo año corrido, 2D 2B “Gran Marina” Peñuelas, Coquimbo | Precio: 600000.0
🏠 PROPIEDAD #4: Se Arrienda Año Corrido 2 Dorm 2 Baños Puertas Del Mar, La Serena. | Precio: 500000.0
🏠 PROPIEDAD #5: Departamento DOÑA ANITA ARRIENDO HASTA DICIEMBRE - LA SERENA | Precio: 750000.0
🏠 PROPIEDAD #6: Departamento SOL DE PEÑUELAS ARRIENDO HASTA DICIEMBRE | Precio: 55

In [3]:
# --- PASO 0: LIMPIEZA TOTAL Y REPARACIÓN ---
import os
import time
import re  # <--- NUEVO: Importamos Regex para limpieza avanzada
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# 1. FORZAR LA PANTALLA PARA noVNC
os.environ["DISPLAY"] = ":99"  

# Cierra procesos viejos
os.system("pkill -9 chrome")
os.system("pkill -9 chromedriver")
os.system("rm -rf /tmp/.com.google.Chrome.*")
os.system("rm -rf /tmp/.org.chromium.Chromium.*")
print("🧹 Limpieza completada. Pantalla virtual configurada.")

# --- VARIABLES GENERALES ---
NOMBRE_GRUPO = "G2"
datos_finales = []   
driver = None        

# --- PASO 1: CONFIGURACIÓN DEL NAVEGADOR ---
options = Options()
# options.add_argument("--headless=new") # Comentado para verlo en noVNC

# Argumentos de estabilidad
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--disable-software-rasterizer")
options.add_argument("--window-size=1920,1080")
options.add_argument(
    "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/120.0.0.0 Safari/537.36"
)

try:
    driver = webdriver.Chrome(options=options)
    print("🚀 Navegador iniciado correctamente.")

    # --- PASO 2: NAVEGACIÓN Y EXTRACCIÓN YAPO ---
    limite_paginas = 3
    url_yapo = "https://www.yapo.cl/bienes-raices-alquiler-apartamentos/chile-es-coquimbo?_gl=1*xrdyl5*_gcl_au*MjQxMDY4NDMuMTc3NjU1NTI1MQ.."
    driver.get(url_yapo)

    for nivel_pagina in range(limite_paginas):
        print(f"\n--- 📄 Procesando Página {nivel_pagina + 1} ---")

        WebDriverWait(driver, 20).until(
            EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".d3-ad-tile__content"))
        )
        
        # --- SCROLL HUMANO ---
        for _ in range(5):
            driver.execute_script("window.scrollBy(0, 800);")
            time.sleep(1)
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(3) 

        bloques = driver.find_elements(By.CSS_SELECTOR, ".d3-ad-tile__content")
        propiedades_extraidas = 0

        for i, bloque in enumerate(bloques):
            try:
                nombre = bloque.find_element(By.CSS_SELECTOR, ".d3-ad-tile__title").get_attribute("textContent")
                precio_texto = bloque.find_element(By.CSS_SELECTOR, ".d3-ad-tile__price").get_attribute("textContent")
                
                if not nombre or not precio_texto or not nombre.strip() or not precio_texto.strip():
                    continue

                try:
                    direccion = bloque.find_element(By.CSS_SELECTOR, ".d3-ad-tile__location").get_attribute("textContent")
                except:
                    direccion = "Dirección no especificada"

                # --- NUEVO: EXTRACCIÓN SEPARADA E INTELIGENTE DE CARACTERÍSTICAS ---
                detalles = bloque.find_elements(By.CSS_SELECTOR, ".d3-ad-tile__details-item")
                
                dormitorios_txt = "0"
                banos_txt = "0"
                estacionamientos_txt = "0"
                
                # Buscamos qué ícono SVG tiene cada detalle para asignarlo correctamente
                for det in detalles:
                    html_interno = det.get_attribute("innerHTML")
                    texto = det.get_attribute("textContent").strip()
                    
                    if "#bed" in html_interno:
                        dormitorios_txt = texto
                    elif "#bath" in html_interno:
                        banos_txt = texto
                    elif "#parking" in html_interno:
                        estacionamientos_txt = texto

                datos_finales.append({
                    "identificador": nombre.strip(),
                    "valor": precio_texto.strip(),
                    "direccion": direccion.strip(),
                    "dormitorios": dormitorios_txt,
                    "estacionamientos": estacionamientos_txt,
                    "banos": banos_txt,
                    "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S"),
                    "grupo": NOMBRE_GRUPO
                })
                propiedades_extraidas += 1
                
            except Exception as e:
                continue
                
        print(f"✅ Se extrajeron {propiedades_extraidas} propiedades reales de {len(bloques)} bloques.")

        try:
            btn_sig = driver.find_element(By.XPATH, "//a[contains(@class, 'pagination') and contains(text(), 'Siguiente')] | //a[contains(@class, 'next')]")
            driver.execute_script("arguments[0].click();", btn_sig)
            time.sleep(5)
        except:
            print("⚠️ No se encontró el botón siguiente o ya es la última página. Terminando paginación.")
            break

    print(f"\n📊 Extracción terminada: {len(datos_finales)} propiedades obtenidas en total.")

except Exception as e:
    print(f"🚨 Error en Selenium: {e}")

finally:
    if driver is not None:
        try:
            driver.quit()
        except:
            pass

# --- PASO 3: GUARDAR EN MONGODB ---
try:
    if datos_finales:
        client = MongoClient("mongodb://mongodb:27017/", serverSelectionTimeoutMS=5000)
        db = client["PruebaYapo"]
        
        # NUEVO: Cambiamos a una colección limpia
        coleccion = db["G3_PruebaRealState_Limpia"] 

        for d in datos_finales:
            # 1. NUEVO: Limpieza del precio INFALIBLE con Regex
            precio_str = str(d["valor"])
            precio_solo_numeros = re.sub(r"\D", "", precio_str) 
            d["valor"] = float(precio_solo_numeros) if precio_solo_numeros else 0.0
            
            # 2. Limpieza y conversión a Enteros
            d["dormitorios"] = int(''.join(filter(str.isdigit, str(d["dormitorios"]))) or 0)
            d["estacionamientos"] = int(''.join(filter(str.isdigit, str(d["estacionamientos"]))) or 0)
            d["banos"] = int(''.join(filter(str.isdigit, str(d["banos"]))) or 0)

        coleccion.insert_many(datos_finales)
        print(f"💾 ¡{len(datos_finales)} datos cargados en la colección '{coleccion.name}' de la BD '{db.name}' correctamente!")
    else:
        print("⚠️ No hay datos válidos para guardar.")

except Exception as e:
    print(f"🚨 Error en MongoDB: {e}")

🧹 Limpieza completada. Pantalla virtual configurada.
🚀 Navegador iniciado correctamente.

--- 📄 Procesando Página 1 ---
✅ Se extrajeron 30 propiedades reales de 30 bloques.

--- 📄 Procesando Página 2 ---
✅ Se extrajeron 30 propiedades reales de 30 bloques.

--- 📄 Procesando Página 3 ---
✅ Se extrajeron 30 propiedades reales de 30 bloques.

📊 Extracción terminada: 90 propiedades obtenidas en total.
💾 ¡90 datos cargados en la colección 'G3_PruebaRealState_Limpia' de la BD 'PruebaYapo' correctamente!


In [3]:
from pyspark.sql import SparkSession

try: 
    spark.stop()
except: 
    pass

print("🚀 Iniciando conexión Spark - MongoDB (Base de datos: PruebaYapo)...")

spark = SparkSession.builder \
    .appName("Analisis_Yapo_Limpio") \
    .config("spark.mongodb.read.connection.uri", "mongodb://mongodb:27017/PruebaYapo.G3_PruebaRealState_Limpia") \
    .getOrCreate()

# Cargamos el DataFrame
df = spark.read.format("mongodb").load()

# --- EL SEGURO ANTI-ERRORES ---
# Verificamos si hay datos ANTES de intentar seleccionar columnas
total_registros = df.count()

if total_registros == 0:
    print("⚠️ ¡ALERTA! La colección está vacía en MongoDB. Debes ejecutar el script de Selenium primero.")
else:
    print(f"✅ Éxito total: {total_registros} registros limpios cargados.")

    # Ordenamos las columnas solo si sabemos que existen
    df_ordenado = df.select(
        "identificador", 
        "valor", 
        "dormitorios", 
        "banos", 
        "estacionamientos", 
        "direccion"
    )

    df_ordenado.show(5, truncate=False)

🚀 Iniciando conexión Spark - MongoDB (Base de datos: PruebaYapo)...
✅ Éxito total: 90 registros limpios cargados.
+--------------------------------------------------------------------+---------+-----------+-----+----------------+---------+
|identificador                                                       |valor    |dormitorios|banos|estacionamientos|direccion|
+--------------------------------------------------------------------+---------+-----------+-----+----------------+---------+
|Arriendo año corrido, amoblado, 2D 2B “Cumbres de Peñuelas” Coquimbo|600000.0 |2          |2    |1               |Coquimbo |
|DEPARTAMENTO Avenida Las Higueras con Anibal Pinto                  |3700005.0|1          |1    |0               |La Serena|
|Arriendo año corrido, 2D 2B “Gran Marina” Peñuelas, Coquimbo        |600000.0 |2          |2    |1               |Coquimbo |
|Departamento PACIFICO 3100 HERMOSO DEPARTAMENTO HASTA DICIEMBRE     |500000.0 |2          |2    |1               |La Serena|
|Dep

In [5]:
from pyspark.sql.functions import col, avg, count, round, format_number, concat, lit

# --- EL FILTRO SALVAVIDAS ---
df_validado = df.filter((col("valor") >= 150000) & (col("valor") <= 1500000))

# --- REPORTE 1: POR DORMITORIOS ---
print("\n📊 Reporte por Dormitorios")
reporte_dorms = df_validado.groupBy("dormitorios").agg(
    count("*").alias("cantidad"),
    avg("valor").alias("precio_prom")
).withColumn(
    "precio_promedio",
    concat(lit("$"), format_number(round(col("precio_prom"), 0), 0))
).orderBy("dormitorios")

reporte_dorms.select("dormitorios", "cantidad", "precio_promedio").show(truncate=False)

# --- REPORTE 2: POR BAÑOS ---
print("\n📊 Reporte por Baños")
reporte_banos = df_validado.groupBy("banos").agg(
    count("*").alias("cantidad"),
    avg("valor").alias("precio_prom")
).withColumn(
    "precio_promedio",
    concat(lit("$"), format_number(round(col("precio_prom"), 0), 0))
).orderBy("banos")

reporte_banos.select("banos", "cantidad", "precio_promedio").show(truncate=False)

# --- REPORTE 3: POR ESTACIONAMIENTOS ---
print("\n📊 Reporte por Estacionamientos")
reporte_estac = df_validado.groupBy("estacionamientos").agg(
    count("*").alias("cantidad"),
    avg("valor").alias("precio_prom")
).withColumn(
    "precio_promedio",
    concat(lit("$"), format_number(round(col("precio_prom"), 0), 0))
).orderBy("estacionamientos")

reporte_estac.select("estacionamientos", "cantidad", "precio_promedio").show(truncate=False)


📊 Reporte por Dormitorios
+-----------+--------+---------------+
|dormitorios|cantidad|precio_promedio|
+-----------+--------+---------------+
|0          |2       |$700,000       |
|1          |2       |$485,000       |
|2          |30      |$572,833       |
|3          |27      |$648,148       |
|4          |1       |$850,000       |
+-----------+--------+---------------+


📊 Reporte por Baños
+-----+--------+---------------+
|banos|cantidad|precio_promedio|
+-----+--------+---------------+
|0    |3       |$606,667       |
|1    |22      |$517,955       |
|2    |36      |$662,222       |
|3    |1       |$850,000       |
+-----+--------+---------------+


📊 Reporte por Estacionamientos
+----------------+--------+---------------+
|estacionamientos|cantidad|precio_promedio|
+----------------+--------+---------------+
|0               |5       |$590,000       |
|1               |53      |$585,189       |
|2               |4       |$985,000       |
+----------------+--------+------------

In [7]:
from pyspark.sql.functions import col, avg, count, round, format_number, concat, lit, when

# 1. Clasificamos las propiedades por Ciudad analizando la dirección
df_ciudades = df.withColumn("ciudad", 
    when(col("direccion").contains("La Serena"), "La Serena")
    .when(col("direccion").contains("Coquimbo"), "Coquimbo")
    .otherwise("Otras zonas")
)

# 2. Aplicamos el filtro de banda de precios realista (evita outliers)
df_validado = df_ciudades.filter((col("valor") >= 150000) & (col("valor") <= 1500000))

# --- REPORTE GENERAL POR COMUNA ---
print("\n📊 Resumen General: La Serena vs Coquimbo")
reporte_comuna = df_validado.groupBy("ciudad").agg(
    count("*").alias("cantidad"),
    avg("valor").alias("precio_prom")
).withColumn(
    "precio_promedio",
    concat(lit("$"), format_number(round(col("precio_prom"), 0), 0))
).orderBy("ciudad")

reporte_comuna.select("ciudad", "cantidad", "precio_promedio").show(truncate=False)

# --- REPORTE DETALLADO: CIUDAD + DORMITORIOS ---
print("\n🏙️ Análisis por Ciudad y Dormitorios")
reporte_dorms_ciudad = df_validado.groupBy("ciudad", "dormitorios").agg(
    count("*").alias("cantidad"),
    avg("valor").alias("precio_prom")
).withColumn(
    "precio_promedio",
    concat(lit("$"), format_number(round(col("precio_prom"), 0), 0))
).orderBy("ciudad", "dormitorios")

reporte_dorms_ciudad.select("ciudad", "dormitorios", "cantidad", "precio_promedio").show(truncate=False)

# --- REPORTE DETALLADO: CIUDAD + BAÑOS ---
print("\n🚿 Análisis por Ciudad y Baños")
reporte_banos_ciudad = df_validado.groupBy("ciudad", "banos").agg(
    count("*").alias("cantidad"),
    avg("valor").alias("precio_prom")
).withColumn(
    "precio_promedio",
    concat(lit("$"), format_number(round(col("precio_prom"), 0), 0))
).orderBy("ciudad", "banos")

reporte_banos_ciudad.select("ciudad", "banos", "cantidad", "precio_promedio").show(truncate=False)

# --- REPORTE DETALLADO: CIUDAD + ESTACIONAMIENTOS ---
print("\n🚗 Análisis por Ciudad y Estacionamientos")
reporte_estac_ciudad = df_validado.groupBy("ciudad", "estacionamientos").agg(
    count("*").alias("cantidad"),
    avg("valor").alias("precio_prom")
).withColumn(
    "precio_promedio",
    concat(lit("$"), format_number(round(col("precio_prom"), 0), 0))
).orderBy("ciudad", "estacionamientos")

reporte_estac_ciudad.select("ciudad", "estacionamientos", "cantidad", "precio_promedio").show(truncate=False)


📊 Resumen General: La Serena vs Coquimbo
+---------+--------+---------------+
|ciudad   |cantidad|precio_promedio|
+---------+--------+---------------+
|Coquimbo |21      |$671,429       |
|La Serena|41      |$580,610       |
+---------+--------+---------------+


🏙️ Análisis por Ciudad y Dormitorios
+---------+-----------+--------+---------------+
|ciudad   |dormitorios|cantidad|precio_promedio|
+---------+-----------+--------+---------------+
|Coquimbo |0          |2       |$700,000       |
|Coquimbo |1          |1       |$450,000       |
|Coquimbo |2          |7       |$517,143       |
|Coquimbo |3          |10      |$778,000       |
|Coquimbo |4          |1       |$850,000       |
|La Serena|1          |1       |$520,000       |
|La Serena|2          |23      |$589,783       |
|La Serena|3          |17      |$571,765       |
+---------+-----------+--------+---------------+


🚿 Análisis por Ciudad y Baños
+---------+-----+--------+---------------+
|ciudad   |banos|cantidad|precio_p

In [9]:
# --- PASO 0: LIMPIEZA TOTAL Y REPARACIÓN ---
import os
import time
import re  
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# 1. FORZAR LA PANTALLA PARA noVNC
os.environ["DISPLAY"] = ":99"  

# Cierra procesos viejos
os.system("pkill -9 chrome")
os.system("pkill -9 chromedriver")
os.system("rm -rf /tmp/.com.google.Chrome.*")
os.system("rm -rf /tmp/.org.chromium.Chromium.*")
print("🧹 Limpieza completada. Pantalla virtual configurada.")

# --- VARIABLES GENERALES ---
NOMBRE_GRUPO = "G2"
datos_finales = []   
driver = None        

# --- PASO 1: CONFIGURACIÓN DEL NAVEGADOR ---
options = Options()
# options.add_argument("--headless=new") # Comentado para verlo en noVNC

# Argumentos de estabilidad
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--disable-software-rasterizer")
options.add_argument("--window-size=1920,1080")
options.add_argument(
    "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/120.0.0.0 Safari/537.36"
)

try:
    driver = webdriver.Chrome(options=options)
    print("🚀 Navegador iniciado correctamente.")

    # --- PASO 2: NAVEGACIÓN Y EXTRACCIÓN YAPO ---
    limite_paginas = 3
    url_yapo = "https://www.yapo.cl/bienes-raices-alquiler-apartamentos/chile-es-coquimbo?_gl=1*xrdyl5*_gcl_au*MjQxMDY4NDMuMTc3NjU1NTI1MQ.."
    driver.get(url_yapo)

    for nivel_pagina in range(limite_paginas):
        print(f"\n--- 📄 Procesando Página {nivel_pagina + 1} ---")

        WebDriverWait(driver, 20).until(
            EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".d3-ad-tile__content"))
        )
        
        # --- SCROLL HUMANO ---
        for _ in range(5):
            driver.execute_script("window.scrollBy(0, 800);")
            time.sleep(1)
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(3) 

        bloques = driver.find_elements(By.CSS_SELECTOR, ".d3-ad-tile__content")
        propiedades_extraidas = 0

        for i, bloque in enumerate(bloques):
            try:
                nombre = bloque.find_element(By.CSS_SELECTOR, ".d3-ad-tile__title").get_attribute("textContent")
                precio_texto = bloque.find_element(By.CSS_SELECTOR, ".d3-ad-tile__price").get_attribute("textContent")
                
                if not nombre or not precio_texto or not nombre.strip() or not precio_texto.strip():
                    continue

                try:
                    direccion = bloque.find_element(By.CSS_SELECTOR, ".d3-ad-tile__location").get_attribute("textContent")
                except:
                    direccion = "Dirección no especificada"

                # ---> NUEVO: EXTRACCIÓN DE LA DESCRIPCIÓN <---
                try:
                    descripcion = bloque.find_element(By.CSS_SELECTOR, ".d3-ad-tile__short-description").get_attribute("textContent")
                except:
                    descripcion = "Sin descripción"

                # --- EXTRACCIÓN SEPARADA E INTELIGENTE DE CARACTERÍSTICAS ---
                detalles = bloque.find_elements(By.CSS_SELECTOR, ".d3-ad-tile__details-item")
                
                dormitorios_txt = "0"
                banos_txt = "0"
                estacionamientos_txt = "0"
                
                # Buscamos qué ícono SVG tiene cada detalle para asignarlo correctamente
                for det in detalles:
                    html_interno = det.get_attribute("innerHTML")
                    texto = det.get_attribute("textContent").strip()
                    
                    if "#bed" in html_interno:
                        dormitorios_txt = texto
                    elif "#bath" in html_interno:
                        banos_txt = texto
                    elif "#parking" in html_interno:
                        estacionamientos_txt = texto

                datos_finales.append({
                    "identificador": nombre.strip(),
                    "descripcion": descripcion.strip(), # <-- Guardamos la descripción limpia
                    "valor": precio_texto.strip(),
                    "direccion": direccion.strip(),
                    "dormitorios": dormitorios_txt,
                    "estacionamientos": estacionamientos_txt,
                    "banos": banos_txt,
                    "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S"),
                    "grupo": NOMBRE_GRUPO
                })
                propiedades_extraidas += 1
                
            except Exception as e:
                continue
                
        print(f"✅ Se extrajeron {propiedades_extraidas} propiedades reales de {len(bloques)} bloques.")

        try:
            btn_sig = driver.find_element(By.XPATH, "//a[contains(@class, 'pagination') and contains(text(), 'Siguiente')] | //a[contains(@class, 'next')]")
            driver.execute_script("arguments[0].click();", btn_sig)
            time.sleep(5)
        except:
            print("⚠️ No se encontró el botón siguiente o ya es la última página. Terminando paginación.")
            break

    print(f"\n📊 Extracción terminada: {len(datos_finales)} propiedades obtenidas en total.")

except Exception as e:
    print(f"🚨 Error en Selenium: {e}")

finally:
    if driver is not None:
        try:
            driver.quit()
        except:
            pass

# --- PASO 3: GUARDAR EN MONGODB ---
try:
    if datos_finales:
        client = MongoClient("mongodb://mongodb:27017/", serverSelectionTimeoutMS=5000)
        db = client["PruebaYapo"]
        
        # NUEVO: Colección G4 para evitar errores de columnas faltantes con PySpark
        coleccion = db["G4_PruebaRealState_Full"] 

        for d in datos_finales:
            # 1. Limpieza del precio INFALIBLE con Regex
            precio_str = str(d["valor"])
            precio_solo_numeros = re.sub(r"\D", "", precio_str) 
            d["valor"] = float(precio_solo_numeros) if precio_solo_numeros else 0.0
            
            # 2. Limpieza y conversión a Enteros
            d["dormitorios"] = int(''.join(filter(str.isdigit, str(d["dormitorios"]))) or 0)
            d["estacionamientos"] = int(''.join(filter(str.isdigit, str(d["estacionamientos"]))) or 0)
            d["banos"] = int(''.join(filter(str.isdigit, str(d["banos"]))) or 0)

        coleccion.insert_many(datos_finales)
        print(f"💾 ¡{len(datos_finales)} datos cargados en la colección '{coleccion.name}' de la BD '{db.name}' correctamente!")
    else:
        print("⚠️ No hay datos válidos para guardar.")

except Exception as e:
    print(f"🚨 Error en MongoDB: {e}")

🧹 Limpieza completada. Pantalla virtual configurada.
🚀 Navegador iniciado correctamente.

--- 📄 Procesando Página 1 ---
✅ Se extrajeron 30 propiedades reales de 30 bloques.

--- 📄 Procesando Página 2 ---
✅ Se extrajeron 30 propiedades reales de 30 bloques.

--- 📄 Procesando Página 3 ---
✅ Se extrajeron 30 propiedades reales de 30 bloques.

📊 Extracción terminada: 90 propiedades obtenidas en total.
💾 ¡90 datos cargados en la colección 'G4_PruebaRealState_Full' de la BD 'PruebaYapo' correctamente!


In [15]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import lower, col, when, count, avg, round, format_number, concat, lit

# --- 1. REINICIO Y CONEXIÓN A LA NUEVA COLECCIÓN G4 ---
try: 
    spark.stop()
except: 
    pass

spark = SparkSession.builder \
    .appName("Analisis_Yapo_TextMining") \
    .config("spark.mongodb.read.connection.uri", "mongodb://mongodb:27017/PruebaYapo.G4_PruebaRealState_Full") \
    .getOrCreate()

# Cargamos el DataFrame con la columna 'descripcion' incluida
df = spark.read.format("mongodb").load()

print(f"✅ Datos cargados desde G4. Total de registros: {df.count()}")

# --- 2. FILTRO DE PRECIOS REALISTAS ---
df_validado = df.filter((col("valor") >= 150000) & (col("valor") <= 1500000))

# --- 3. TEXT MINING ---
# Convertimos textos a minúsculas
df_text_mining = df_validado.withColumn("desc_lower", lower(col("descripcion"))) \
                            .withColumn("titulo_lower", lower(col("identificador")))

# Creamos las variables dummy
df_amenities = df_text_mining.withColumn("tiene_gimnasio", 
    when(col("desc_lower").contains("gimnasio") | col("titulo_lower").contains("gimnasio") | col("desc_lower").contains("gym"), 1).otherwise(0)
).withColumn("tiene_quinchos", 
    when(col("desc_lower").contains("quinchos") | col("titulo_lower").contains("quinchos") | col("desc_lower").contains("quinchos"), 1).otherwise(0)
).withColumn("tiene_lavanderia", 
    when(col("desc_lower").contains("lavanderia") | col("desc_lower").contains("lavandería"), 1).otherwise(0)
).withColumn("areas_verdes", 
    when(col("desc_lower").contains("areas verdes") | col("desc_lower").contains("áreas verdes") | col("desc_lower").contains("jardin"), 1).otherwise(0)
)

# --- 4. REPORTES ---
print("\n🏋️ Impacto del Gimnasio en el Precio:")
df_amenities.groupBy("tiene_gimnasio").agg(
    count("*").alias("cantidad"),
    avg("valor").alias("precio_prom")
).withColumn(
    "precio_promedio",
    concat(lit("$"), format_number(round(col("precio_prom"), 0), 0))
).orderBy("tiene_gimnasio").show()

print("\n Impacto del Quincho en el Precio:")
df_amenities.groupBy("tiene_quinchos").agg(
    count("*").alias("cantidad"),
    avg("valor").alias("precio_prom")
).withColumn(
    "precio_promedio",
    concat(lit("$"), format_number(round(col("precio_prom"), 0), 0))
).orderBy("tiene_quinchos").show()

✅ Datos cargados desde G4. Total de registros: 90

🏋️ Impacto del Gimnasio en el Precio:
+--------------+--------+-----------------+---------------+
|tiene_gimnasio|cantidad|      precio_prom|precio_promedio|
+--------------+--------+-----------------+---------------+
|             0|      65|613153.8461538461|       $613,154|
|             1|       1|         750000.0|       $750,000|
+--------------+--------+-----------------+---------------+


 Impacto del Quincho en el Precio:
+--------------+--------+-----------------+---------------+
|tiene_quinchos|cantidad|      precio_prom|precio_promedio|
+--------------+--------+-----------------+---------------+
|             0|      66|615227.2727272727|       $615,227|
+--------------+--------+-----------------+---------------+

